In [ ]:
!pip uninstall -y pymupdf fitz
!pip install -q pymupdf==1.24.10
!pip install -q doclayout-yolo ultralytics

import os
os.kill(os.getpid(), 9)

In [ ]:
import fitz
from pydantic import BaseModel
from typing import List
from doclayout_yolo import YOLOv10
import re
import numpy as np
from PIL import Image

In [ ]:
class ParsedPage(BaseModel):
    page_index: int
    text: str
    raw_blocks: list = None
    layout_boxes: list = None


class ParsedPaper(BaseModel):
    paper_id: str
    pdf_path: str
    num_pages: int
    pages: list[ParsedPage]

In [ ]:
def clean_page_texts(pages: list[ParsedPage]) -> list[ParsedPage]:
    # 1. Detect header/footer candidates
    first_lines = [p.text.splitlines()[0] if p.text.splitlines() else '' for p in pages]
    last_lines = [p.text.splitlines()[-1] if p.text.splitlines() else '' for p in pages]
    header_counts = {}
    footer_counts = {}
    for line in first_lines:
        header_counts[line] = header_counts.get(line, 0) + 1
    for line in last_lines:
        footer_counts[line] = footer_counts.get(line, 0) + 1
    header = max(header_counts, key=header_counts.get) if header_counts else ''
    footer = max(footer_counts, key=footer_counts.get) if footer_counts else ''
    header_thresh = len(pages) // 2
    footer_thresh = len(pages) // 2

    cleaned_pages = []
    for p in pages:
        lines = p.text.splitlines()
        # Remove header/footer if repeated
        if lines and header and lines[0] == header and header_counts[header] > header_thresh:
            lines = lines[1:]
        if lines and footer and lines[-1] == footer and footer_counts[footer] > footer_thresh:
            lines = lines[:-1]
        # Hyphenation fix & join lines
        new_lines = []
        i = 0
        while i < len(lines):
            line = lines[i]
            if line.rstrip().endswith('-') and i+1 < len(lines):
                next_line = lines[i+1].lstrip()
                if next_line and next_line[0].islower():
                    line = line.rstrip()[:-1] + next_line
                    i += 1
            new_lines.append(line)
            i += 1
        # Join lines, fix linebreaks
        text = '\n'.join(new_lines)
        text = re.sub(r'(?<![.!?:])\n([a-z])', r' \1', text)
        text = re.sub(r'\s{2,}', ' ', text)
        cleaned_pages.append(ParsedPage(page_index=p.page_index, text=text.strip()))
    return cleaned_pages

In [ ]:
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10

model_path = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt"
)
print(f"Model downloaded at: {model_path}")

def parse_pdf_to_pages(pdf_path: str, paper_id: str) -> ParsedPaper:
    """Parse PDF + DocLayout-YOLO - ĐÃ SỬA"""
    doc = fitz.open(pdf_path)
    pages: list[ParsedPage] = []
    
    print("Loading model...")
    model = YOLOv10(model_path)   

    for page_index in range(doc.page_count):
        page = doc.load_page(page_index)
        
        plain_text = page.get_text("text")
        raw_dict = page.get_text("dict")

        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        img = pix.tobytes("png")   
        
        img_pil = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img_np = np.array(img_pil)

        result = model.predict(img_np, imgsz=1024, verbose=False)[0]

        pages.append(ParsedPage(
            page_index=page_index,
            text=plain_text,
            raw_blocks=raw_dict["blocks"],
            layout_boxes=result.boxes if result else []
        ))

    doc.close()
    print(f"Parse xong {len(pages)} trang")
    return ParsedPaper(
        paper_id=paper_id,
        pdf_path=pdf_path,
        num_pages=len(pages),
        pages=pages,
    )

In [ ]:
def detect_sections(pages: list[ParsedPage]) -> list[dict]:
    segments = []
    current_section = "Title and Authors"
    buffer = []

    for page in pages:
        if not page.layout_boxes:
            # Fallback mạnh
            page_text = page.text.strip()
            page_segments = _fallback_detect(page_text, page.page_index, current_section)
            segments.extend(page_segments)
            if page_segments:
                current_section = page_segments[-1]["section"]
            continue

        for box in page.layout_boxes:
            cls_name = str(getattr(box, 'cls_name', getattr(box, 'cls', ''))).lower()
            text_in_box = _extract_text_from_box(page, box).strip()

            if text_in_box:
                # DEBUG
                print(f"DEBUG Page {page.page_index} | Class: {cls_name} | Text: {text_in_box[:100]}")

                if _is_strong_heading(text_in_box):
                    if buffer:
                        segments.append({
                            "section": current_section,
                            "page_index": page.page_index,
                            "text": " ".join(buffer).strip()
                        })
                        buffer = []
                    current_section = text_in_box
                else:
                    buffer.append(text_in_box)

        if buffer:
            segments.append({
                "section": current_section,
                "page_index": page.page_index,
                "text": " ".join(buffer).strip()
            })
            buffer = []

    return segments


def _is_strong_heading(text: str) -> bool:
    """Logic detect heading cực mạnh"""
    if not text or len(text) > 180:
        return False

    text_upper = text.upper().strip()

    # Common sections
    if any(word in text_upper for word in [
        "ABSTRACT", "INTRODUCTION", "RELATED WORK", "RELATED WORKS",
        "MEMORY OPTIMIZATION", "TRADE COMPUTATION", "EXPERIMENTS", 
        "RESULTS", "DISCUSSION", "CONCLUSION", "ACKNOWLEDGMENT", "REFERENCES"
    ]):
        return True

    # Numbered headings
    if re.match(r'^\s*\d+(\.\d+)*\s+[A-Z]', text):
        return True

    # Short uppercase or title case
    words = text.split()
    if 2 <= len(words) <= 12 and (text.isupper() or sum(w[0].isupper() for w in words if w) / len(words) > 0.6):
        return True

    return False


def _fallback_detect(text: str, page_index: int, current: str) -> list[dict]:
    """Fallback khi không có layout"""
    segments = []
    parts = re.split(r'(?=\b(?:ABSTRACT|INTRODUCTION|RELATED|MEMORY|TRADE|EXPERIMENTS?|CONCLUSION|ACKNOWLEDGMENT|REFERENCES?)\b)', text, flags=re.I)
    
    for part in parts:
        stripped = part.strip()
        if not stripped:
            continue
        if _is_strong_heading(stripped):
            segments.append({
                "section": current,
                "page_index": page_index,
                "text": stripped
            })
            current = stripped

    return segments

In [ ]:
paper = parse_pdf_to_pages("/content/paper_001.pdf", "paper001")
cleaned = clean_page_texts(paper.pages)
sections = detect_sections(cleaned)

for sec in sections[:20]:
    print(f"Page {sec['page_index']:2d} | {sec['section'][:85]:85} | {len(sec['text'])} chars")